In [2]:
!pip install hqq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hqq: filename=hqq-0.2.8.post1-py3-none-any.whl size=68503 sha256=68baf200730426cb5e2e7caf67a89f2e031f2666175ec2c6fd0cf458efcf9e73
  Stored in directory: /root/.cache/pip/wheels/c9/a7/fd/d254c051dd85ba5a3c4a27bfc2da125647df14ee35f0d8d892
Successfully built hqq


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
import os

os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
# Replace with your username)
os.environ["HF_USERNAME"] = "kabirvats"

In [4]:
import os
import csv
import json
import random
import string
from time import perf_counter

import torch
from matplotlib import pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["TOKENIZERS_PARALLELISM"] = "0"

In [5]:
def _safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)

def save_line_chart(title, x, y, ylabel, xlabel, output_path):
    plt.figure()
    plt.plot(x, y, marker="o")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

def normalize_text(s: str) -> str:
    return " ".join(s.strip().split()).lower()

def exact_match(pred: str, gold: str) -> int:
    return int(normalize_text(pred) == normalize_text(gold))

def substring_match(pred: str, gold: str) -> int:
    return int(normalize_text(gold) in normalize_text(pred))

def random_code(k=8):
    alphabet = string.ascii_uppercase + string.digits
    return "".join(random.choice(alphabet) for _ in range(k))

def make_needle(i: int) -> str:
    # keep it easy to verify exactly
    return f"NEEDLE_{i}_{random_code(10)}"

def make_haystack_unit():
    return (
        "The archive contains ordinary details about villages, markets, weather, "
        "bridges, songs, tools, road dust, and farming schedules. "
    )

def make_haystack_with_at_least_n_tokens(tokenizer, n_tokens: int) -> str:
    text = ""
    unit = make_haystack_unit()
    while True:
        text += unit
        tok_len = len(tokenizer(text, add_special_tokens=False)["input_ids"])
        if tok_len >= n_tokens:
            return text

def trim_to_token_length(tokenizer, text: str, max_tokens: int) -> str:
    ids = tokenizer(text, add_special_tokens=False)["input_ids"][:max_tokens]
    return tokenizer.decode(ids, skip_special_tokens=True)

In [6]:
def insert_needle_by_depth(tokenizer, haystack: str, needle: str, depth: float, total_tokens: int) -> str:
    needle_sentence = f"\n\nImportant fact: The secret passcode is {needle}.\n\n"
    needle_len = len(tokenizer(needle_sentence, add_special_tokens=False)["input_ids"])

    haystack_budget = max(0, total_tokens - needle_len)
    haystack = trim_to_token_length(tokenizer, haystack, haystack_budget)

    ids = tokenizer(haystack, add_special_tokens=False)["input_ids"]
    insert_at = int(depth * len(ids)) if len(ids) else 0

    left = tokenizer.decode(ids[:insert_at], skip_special_tokens=True)
    right = tokenizer.decode(ids[insert_at:], skip_special_tokens=True)

    combined = left + needle_sentence + right
    return trim_to_token_length(tokenizer, combined, total_tokens)

def build_prompt(context: str) -> str:
    return (
        "Read the context and answer the question with only the exact passcode.\n\n"
        f"Context:\n{context}\n\n"
        "Question: What is the secret passcode?\n"
        "Answer:"
    )

In [7]:
def build_needle_dataset(tokenizer, model, context_lengths, depths, trials_per_setting=3, extra_buffer_tokens=128, answer_budget=20):
    rows = []

    overhead = len(tokenizer(build_prompt(""), add_special_tokens=False)["input_ids"])
    model_max_len = getattr(model.config, "max_position_embeddings", 2048)

    for requested_length in context_lengths:
        total_prompt_budget = min(requested_length, model_max_len)
        context_budget = total_prompt_budget - overhead - answer_budget
        if context_budget <= 0:
            continue

        base_haystack = make_haystack_with_at_least_n_tokens(
            tokenizer, context_budget + extra_buffer_tokens
        )

        for depth in depths:
            for trial_idx in range(trials_per_setting):
                needle = make_needle(trial_idx)
                context = insert_needle_by_depth(
                    tokenizer=tokenizer,
                    haystack=base_haystack,
                    needle=needle,
                    depth=depth,
                    total_tokens=context_budget,
                )
                prompt = build_prompt(context)

                rows.append({
                    "context_length": requested_length,
                    "actual_prompt_tokens": len(tokenizer(prompt, add_special_tokens=False)["input_ids"]),
                    "depth": depth,
                    "trial_idx": trial_idx,
                    "needle": needle,
                    "prompt": prompt,
                })

    return Dataset.from_list(rows)

In [8]:
@torch.inference_mode()
def generate_answer(
    model,
    tokenizer,
    prompts,
    max_new_tokens=24,
    deterministic=True,
    cache_implementation="dynamic",
    cache_backend="quanto",
    nbits=4,
):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,          # do not silently hide the problem
        add_special_tokens=False,  # keeps budgeting cleaner
    ).to(model.device)

    seq_len = inputs["input_ids"].shape[1]
    limit = getattr(model.config, "max_position_embeddings", None)

    if limit is not None:
        if seq_len >= limit:
            raise ValueError(f"Prompt length {seq_len} >= model limit {limit}")

        safe_new_tokens = min(max_new_tokens, limit - seq_len)
        if safe_new_tokens <= 0:
            raise ValueError(
                f"No room for generation: seq_len={seq_len}, limit={limit}"
            )
    else:
        safe_new_tokens = max_new_tokens

    gen_kwargs = dict(
        max_new_tokens=safe_new_tokens,
        return_dict_in_generate=True,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

    if deterministic:
        gen_kwargs.update(dict(do_sample=False))
    else:
        gen_kwargs.update(dict(do_sample=True, temperature=0.7, top_p=0.9))

    if cache_implementation == "quantized":
        cache_config = {
            "backend": cache_backend,
            "nbits": nbits,
        }

        if cache_backend == "quanto":
            cache_config["axis_key"] = 0
            cache_config["axis_value"] = 0
        elif cache_backend == "hqq":
            cache_config["axis_key"] = 1
            cache_config["axis_value"] = 1
        else:
            raise ValueError(f"Unknown cache_backend: {cache_backend}")

        gen_kwargs["cache_implementation"] = "quantized"
        gen_kwargs["cache_config"] = cache_config

    print(
        f"batch_seq_len={seq_len}, safe_new_tokens={safe_new_tokens}, "
        f"limit={limit}, overflow={seq_len + safe_new_tokens > limit if limit else False}"
    )

    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = perf_counter()
    outputs = model.generate(**inputs, **gen_kwargs)
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    elapsed = perf_counter() - t0

    sequences = outputs.sequences
    input_width = inputs["input_ids"].shape[1]  # padded batch width

    new_texts = []
    for i in range(sequences.shape[0]):
        new_tokens = sequences[i, input_width:]
        new_texts.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

    return new_texts, elapsed

In [9]:
def save_results_csv(output_dir, rows, filename="needle_results.csv"):
    _safe_mkdir(output_dir)
    csv_path = os.path.join(output_dir, filename)

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "context_length",
                "depth",
                "trial_idx",
                "needle",
                "prediction",
                "exact_match",
                "substring_match",
                "latency_sec",
                "model_name",
            ],
        )
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

    return csv_path

def aggregate_for_plot(rows, x_key, fixed_key):
    """
    Example:
      x_key="depth", fixed_key="context_length"
      => one curve per context_length
    """
    grouped = {}
    for r in rows:
        curve_name = r[fixed_key]
        x = r[x_key]
        grouped.setdefault(curve_name, {}).setdefault(x, []).append(r["substring_match"])

    curves = {}
    for curve_name, values in grouped.items():
        xs = sorted(values.keys())
        ys = [sum(values[x]) / len(values[x]) for x in xs]
        curves[curve_name] = (xs, ys)
    return curves

def save_multi_curve_plot(curves, title, xlabel, ylabel, output_path):
    plt.figure()
    for curve_name, (xs, ys) in sorted(curves.items(), key=lambda kv: kv[0]):
        plt.plot(xs, ys, marker="o", label=str(curve_name))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.ylim(-0.05, 1.05)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.legend(title="Curve")
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

In [10]:
def needle_in_haystack_benchmark(
    model,
    tokenizer,
    dataset,
    output_dir,
    batch_size=1,
    max_new_tokens=12,
    deterministic=True,
    cache_implementation="dynamic",
    cache_backend="quanto",
    nbits=4,
):
    _safe_mkdir(output_dir)

    # warmup
    warm = tokenizer(["hello world"] * 2, return_tensors="pt", padding=True).to(model.device)
    for _ in range(2):
        _ = model.generate(**warm, max_new_tokens=8, do_sample=False)

    rows = []
    model_name = getattr(getattr(model, "config", None), "_name_or_path", None)

    for start in range(0, len(dataset), batch_size):
        batch = dataset[start:start + batch_size]
        prompts = batch["prompt"]
        needles = batch["needle"]
        context_lengths = batch["context_length"]
        depths = batch["depth"]
        trial_idxs = batch["trial_idx"]

        preds, elapsed = generate_answer(
            model=model,
            tokenizer=tokenizer,
            prompts=prompts,
            max_new_tokens=max_new_tokens,
            deterministic=deterministic,
            cache_implementation=cache_implementation,
            cache_backend=cache_backend,
            nbits=nbits,
        )

        per_sample_latency = elapsed / max(1, len(prompts))

        for i in range(len(prompts)):
            pred = preds[i]
            gold = needles[i]
            em = exact_match(pred, gold)
            sm = substring_match(pred, gold)

            rows.append(
                {
                    "context_length": int(context_lengths[i]),
                    "depth": float(depths[i]),
                    "trial_idx": int(trial_idxs[i]),
                    "needle": gold,
                    "prediction": pred,
                    "exact_match": em,
                    "substring_match": sm,
                    "latency_sec": per_sample_latency,
                    "model_name": model_name,
                }
            )

            print("\n=== SAMPLE ===")
            print("context_length:", context_lengths[i])
            print("depth:", depths[i])
            print("gold:", gold)
            print("pred:", pred)
            print("exact_match:", em)
            print("substring_match:", sm)

    csv_path = save_results_csv(output_dir, rows)

    # Plot 1: accuracy vs depth, one curve per context length
    curves_depth = aggregate_for_plot(rows, x_key="depth", fixed_key="context_length")
    save_multi_curve_plot(
        curves=curves_depth,
        title="Needle Retrieval Accuracy vs Depth",
        xlabel="Needle depth (0=start, 1=end)",
        ylabel="Exact match rate",
        output_path=os.path.join(output_dir, "accuracy_vs_depth.png"),
    )

    # Plot 2: accuracy vs context length, one curve per depth
    curves_len = aggregate_for_plot(rows, x_key="context_length", fixed_key="depth")
    save_multi_curve_plot(
        curves=curves_len,
        title="Needle Retrieval Accuracy vs Context Length",
        xlabel="Context length (tokens)",
        ylabel="Exact match rate",
        output_path=os.path.join(output_dir, "accuracy_vs_context_length.png"),
    )

    # Plot 3: latency vs context length
    lat_grouped = {}
    for r in rows:
        lat_grouped.setdefault(r["context_length"], []).append(r["latency_sec"])
    xs = sorted(lat_grouped.keys())
    ys = [sum(lat_grouped[x]) / len(lat_grouped[x]) for x in xs]
    save_line_chart(
        title="Needle Benchmark Latency",
        x=xs,
        y=ys,
        ylabel="Seconds / sample",
        xlabel="Context length (tokens)",
        output_path=os.path.join(output_dir, "latency_vs_context_length.png"),
    )

    # summary json
    summary = {
        "model_name": model_name,
        "num_samples": len(rows),
        "mean_exact_match": sum(r["exact_match"] for r in rows) / max(1, len(rows)),
        "mean_substring_match": sum(r["substring_match"] for r in rows) / max(1, len(rows)),
        "context_lengths": sorted(list({r["context_length"] for r in rows})),
        "depths": sorted(list({r["depth"] for r in rows})),
        "max_new_tokens": max_new_tokens,
        "deterministic": deterministic,
    }
    with open(os.path.join(output_dir, "needle_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\nSaved results to: {csv_path}")
    print(json.dumps(summary, indent=2))

In [13]:
#model_name = "Qwen/Qwen2.5-0.5B"
model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
# sensible scale for a 0.5B model
context_lengths = [512, 1024, 1536, 2048]
depths = [0.1, 0.3, 0.5, 0.7, 0.9]


dataset = build_needle_dataset(
    tokenizer=tokenizer,
    context_lengths=context_lengths,
    depths=depths,
    trials_per_setting=3,
    model=model,
)

needle_in_haystack_benchmark(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    output_dir="/kaggle/working/needle_benchmark",
    batch_size=1,
    max_new_tokens=20,
    deterministic=True,
    cache_implementation="quantized",
    cache_backend="hqq",
    nbits=4,
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


batch_seq_len=493, safe_new_tokens=20, limit=2048, overflow=False

=== SAMPLE ===
context_length: 512
depth: 0.1
gold: NEEDLE_0_8UJ9CDUR3A
pred: NEEDLE_0_8UJ9CDUR3A
exact_match: 1
substring_match: 1
batch_seq_len=493, safe_new_tokens=20, limit=2048, overflow=False

=== SAMPLE ===
context_length: 512
depth: 0.1
gold: NEEDLE_1_EOT51VB0NV
pred: NEEDLE_1_EOT51VB0NV
exact_match: 1
substring_match: 1
batch_seq_len=493, safe_new_tokens=20, limit=2048, overflow=False

=== SAMPLE ===
context_length: 512
depth: 0.1
gold: NEEDLE_2_HZ9BV0VBKA
pred: NEEDLE_2_HZ9BV0VBKA
exact_match: 1
substring_match: 1
batch_seq_len=493, safe_new_tokens=20, limit=2048, overflow=False

=== SAMPLE ===
context_length: 512
depth: 0.3
gold: NEEDLE_0_IQBOJUVOHA
pred: NEEDLE_0_IQBOJUVOHA.
exact_match: 0
substring_match: 1
batch_seq_len=493, safe_new_tokens=20, limit=2048, overflow=False

=== SAMPLE ===
context_length: 512
depth: 0.3
gold: NEEDLE_1_XZJ3WERP7T
pred: NEEDLE_1_XZJ3WERP7T
exact_match: 1
substring_match: 1
batc